In [14]:
from pathlib import Path

# Project root = folder containing the notebooks/ directory
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

In [2]:
# Import libraries
import pandas as pd
import numpy as np
import joblib
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, QED

In [7]:
# Load tuned RF model
best_rf = joblib.load(MODELS_DIR/"final_random_forest.pkl")
print("Tuned RF model loaded successfully.")

Tuned RF model loaded successfully.


In [8]:
# Load Coconut database
new_df = pd.read_excel(DATA_DIR/"Virtual Screening"/"COCONUT_dataset.xlsx")
new_df = new_df.loc[:, ~new_df.columns.str.contains("^Unnamed")]

print(f"Total compounds: {len(new_df)}")

Total compounds: 738844


In [9]:
# Generate Morgan fingerprints
def morgan_fp(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return np.nan

    fp = AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius=2,
        nBits=2048
    )

    return np.array(fp, dtype=np.int8)

new_df["FP"] = new_df["Smiles"].apply(morgan_fp)

In [10]:
# Remove invalid SMILES
new_df = new_df.dropna(subset=["FP"]).reset_index(drop=True)

print(f"Valid compounds: {len(new_df)}")

Valid compounds: 738823


In [11]:
# Create prediction matrix
X_new = np.stack(new_df["FP"].values)

In [12]:
# Predict BTK activity
new_df["Predicted_Class"] = best_rf.predict(X_new)
new_df["Probability_Active"] = best_rf.predict_proba(X_new)[:, 1]

new_df["Predicted_Activity"] = new_df["Predicted_Class"].map({
    1: "Active",
    0: "Inactive"
})
new_df.to_excel(
    r"C:\Users\shrut\Downloads\TunedRF_CoconutPred.xlsx",
    index=False
)

print("Predictions saved successfully.")

Predictions saved successfully.


In [8]:
# Load top 100 predicted compounds
top100 = pd.read_excel(
    DATA_DIR/"Virtual Screening\COCONUT_predicted_dataset.xlsx"
)

In [9]:
# Calculate molecular descriptors
top100["Mol"] = top100["Smiles"].apply(Chem.MolFromSmiles)
top100 = top100[top100["Mol"].notnull()].copy()
top100["MW"] = top100["Mol"].apply(Descriptors.MolWt)
top100["LogP"] = top100["Mol"].apply(Descriptors.MolLogP)
top100["TPSA"] = top100["Mol"].apply(Descriptors.TPSA)
top100["HBA"] = top100["Mol"].apply(Descriptors.NumHAcceptors)
top100["HBD"] = top100["Mol"].apply(Descriptors.NumHDonors)
top100["RB"] = top100["Mol"].apply(Descriptors.NumRotatableBonds)

In [10]:
# Calculate Lipinski violations
def lipinski_violations(row):
    violations = 0

    if row["MW"] > 500:
        violations += 1
    if row["LogP"] > 5:
        violations += 1
    if row["HBA"] > 10:
        violations += 1
    if row["HBD"] > 5:
        violations += 1

    return violations

top100["Lipinski_Violations"] = top100.apply(
    lipinski_violations,
    axis=1
)

In [11]:
# Apply drug-likeness filters
filtered = top100[
    (top100["MW"] >= 250) &
    (top100["MW"] <= 550) &
    (top100["TPSA"] <= 140) &
    (top100["RB"] <= 10) &
    (top100["Lipinski_Violations"] <= 1)
].copy()

print(f"Remaining compounds: {len(filtered)}")

Remaining compounds: 62


In [13]:
# Calculate QED scores and filter final compounds
filtered["QED"] = filtered["Mol"].apply(QED.qed)

filtered = filtered[filtered["QED"] >= 0.5].copy()
print(f"Final compounds after QED filtering: {len(filtered)}")
# Save final filtered compounds
filtered.to_excel(
    r"C:\Users\shrut\Downloads\TunedRF_DockingCandidates.xlsx",
    index=False
)

print("Final filtered compounds saved successfully.")

Final compounds after QED filtering: 33
Final filtered compounds saved successfully.
